# MOV06 · M10 — 군수품 이동 상태 추적 및 도착 예정 시간(ETA) 예측 시스템 최종 완성

**학생용 독립 미션 Notebook · 교관 정답 미포함 | 필수 | CMS 4시간**

## 0. 시작 전에
1. 이 파일을 Google Colab에서 엽니다. 첫 번째 코드 셀부터 순서대로 실행하세요.
2. 실습 자료는 준비 셀에 내장되어 있으므로 별도 ZIP을 업로드하지 않습니다.
3. 이 파일은 한 미션의 작성 도구이며, 이전 미션의 실제 산출물이 자동으로 이어졌다고 가정하지 마세요.
4. 실습 중 작성한 코드·대화·보고서는 백업해야 합니다. 실습용 PASS와 교관 승인은 다릅니다.

### 오늘 할 일
이 미션에서는 **교육용 가상 물류 데이터**를 사용해 핵심 함수를 직접 구현하고 실제 실행 결과를 검증합니다. 아래의 **미션 목표 → 자료 준비 → 구현 → 실제 실행 → 근거 기록** 순서로 진행하세요.

- 미션별 자료: `MOV06_m10_student.ipynb`, `MOV06_M10_lesson_20261104v1.md`, `MOV06_M10_forms.md`
- 첫 실행과 수정 실행의 결과 및 AI 대화 원문을 보존하세요.
- 미실행·기준 미달은 있는 그대로 기록하고, 실행 성공과 학습 PASS·교관 승인을 구분하세요.
- 핵심 함수의 정답 구현은 학생이 직접 작성합니다.


## 미션 10 | 수행 흐름과 학습 목표

**학생용 실습 · 정답 구현은 직접 작성**

### 상황과 해결할 문제
분석 코드를 묶는 것만으로 다음 사람이 사용할 도구가 완성되지는 않습니다. 새 이동 자료의 재학습, 승인 대기, 7일 납품 소요 예측과 운영 문서를 함께 점검합니다. 여러분은 교육용 가상 물류 데이터 분석 담당자입니다. 실제 부대 위치를 추적하거나 수송을 지시하는 활동은 아닙니다. 드리프트는 새 자료에서 기존 모델의 오차가 달라지는 상태입니다. 챔피언은 기존 모델, 챌린저는 교체 후보이며, 승격은 실습 코드의 모델 교체 판단이지 실제 담당자의 승인이 아닙니다.

### 이번 미션에서 확인할 목표
신규 이동 40개를 추가해 정합성을 검증하고 기존 모델의 오차로 드리프트를 판단한 뒤 챌린저 모델을 학습해 개선이 3% 이상일 때만 승격한다. 120분 시점 선단의 ETA와 지연·이탈·정차 경보를 산출하고 경보 건은 자동 지시 없이 승인대기로 분리하며, 요일·추세 Ridge로 향후 7일 납품 소요를 예측해 전주 동요일 기준과 비교합니다. 운영 문서 완비율, 인터페이스 명세, 자체 점검표, 브리핑 요약을 출력합니다.

### 구현할 함수의 약속
입출력 계약은 함수가 어떤 자료를 받아 어떤 결과를 내는지 정한 약속입니다. retrain_pipeline: old_end,new_end,hold_n,drift_mape,min_gain → champ_mape/drift/champ_mae/chall_mae/promote/model 사전. 챌린저 학습은 보류 구간 이전까지만 하고 최종 평가 이동은 재학습에서 제외한다.

### 실습 시작 순서
1. **실습자료 자동 준비** 셀을 실행해 교육용 프로젝트가 생성됐는지 확인합니다.
2. **구현 함수 저장** 및 **작업 기록·JSON 백업/복원** 도구를 준비합니다.
3. 아래의 미션 설명과 Lesson·Forms를 읽고 실제 구현·수정·재실행을 합니다.
4. 실제 질문·답변 원문, 최초·수정 코드, 검증 결과를 보존하고 검토용 ZIP을 저장합니다.

### 네 가지 활동 — 순서별 수행·AI 활용·검증

**활동 1. 재학습 시간선과 통합 점검 계획**
- 할 일: 같은 번호의 학습 안내와 빈 작성 양식을 읽고 입력·출력, 자료를 알 수 있는 시점과 원본 평가 조건을 적으세요. 드리프트는 새 자료에서 기존 모델의 오차가 달라지는 상태입니다. 챔피언은 기존 모델, 챌린저는 교체 후보이며, 승격은 실습 코드의 모델 교체 판단이지 실제 담당자의 승인이 아닙니다.
- AI 질문 예시: 기존 이동, 신규 40개 이동, 독립 평가 20개 이동의 시간선과 파이프라인 계약을 붙일게요. 드리프트 임계값 8.0·5.0, 모델 승격 기준 3%, 승인대기·7일 납품 소요 예측의 검증 순서를 계획해 주세요.
- 확인과 기록: `forms/MOV06_M10_forms.md`에 근거·실행값·수정 이유를 적고 원본 출력과 대조합니다.

**활동 2. 재학습 함수와 드리프트 비교 구현하기**
- 할 일: Notebook에서 retrain_pipeline 함수의 입출력 계약을 확인하고 직접 구현·저장하세요. 첫 실행의 코드와 출력 사본을 보존한 뒤 다음 변경을 별도 버전으로 수행합니다: drift_mape를 8.0과 5.0으로 비교하고 드리프트 판단과 3% 승격 정책을 구분한다. 신규 40개와 독립 평가 20개 시간선, 승인 대기, 7일 소요 예측과 운영 문서 12종 본문을 완성한다.
- AI 질문 예시: retrain_pipeline 빈 함수와 단계별 입력을 붙일게요. 시간선에 따라 학습·평가를 분리하고 드리프트 판단 및 승격 후보를 기록하는 코드를 구현할 순서를 설명해 주세요. 평가 데이터 재학습과 자동 승인·집행을 금지해 주세요.
- 확인과 기록: `forms/MOV06_M10_forms.md`에 근거·실행값·수정 이유를 적고 원본 출력과 대조합니다.

**활동 3. 독립 평가·승인 대기·소요 예측 검증하기**
- 할 일: 보류 구간이 학습에 들어가지 않았는지, 재학습·자체 점검·운영 문서 100% 완비를 실제 결과로 확인합니다. 7일 소요 예측은 전주 동요일 기준과 비교하고 경보는 자동 집행 없이 승인대기로 남깁니다. 기존 학습 0~199, 추가 학습 200~229, 모델 선택 230~239, 독립 최종 평가 240~259로 총 260개 이동을 사용합니다. 원본 신규 40개 이동(200~239)에 최종 검증용 20개를 추가한 출제자 보완입니다. 드리프트와 챌린저 승격은 별도 판정이며 원본은 드리프트가 없어도 후보를 학습합니다. 문서 12종은 실제 파일·본문·미작성 표시를 확인합니다. 운영 문서가 미작성 상태라면 최초 점검에서 FAIL이 나올 수 있습니다. 학생은 실제 문서 본문을 작성·저장한 뒤 같은 조건으로 다시 실행하고, 문서 내용의 적합성은 별도 검토를 받아야 합니다.
- AI 질문 예시: 재학습 로그, 독립 평가 20개 이동 결과와 7일 소요 예측을 붙일게요. 드리프트 8.0·5.0 비교, 3% 승격 기준, 승인대기 상태와 예측·실측의 차이를 검증해 주세요. 실행 성공과 배포 승인 PASS를 분리해 주세요.
- 확인과 기록: `forms/MOV06_M10_forms.md`에 근거·실행값·수정 이유를 적고 원본 출력과 대조합니다.

**활동 4. 운영 문서 12종과 브리핑 완성하기**
- 할 일: 지정 산출물과 작성 양식에 첫 실행·수정 실행의 값, 채택·수정·보류 이유를 기록합니다. 실제 대화 원문과 로그·표·그림의 위치도 연결해 저장하세요.
- AI 질문 예시: 운영 문서 12종, 브리핑, 파이프라인 결과표를 붙일게요. 파일별 증거와 미작성 항목, 7일 예측 근거, 승인대기 상태를 확인해 최종 점검표를 만들도록 도와주세요. 직접 실행하지 않은 항목은 PASS로 쓰지 마세요.
- 확인과 기록: `forms/MOV06_M10_forms.md`에 근거·실행값·수정 이유를 적고 원본 출력과 대조합니다.

### 평가·제출 구분
- **실행 성공**은 코드가 오류 없이 끝난 상태이며 평가 기준 충족·교관 승인은 아닙니다.
- 목표 미달, 미실행, 미측정은 그대로 기록합니다. 결과가 없는 상태를 임의 PASS 처리하지 않습니다.
- 자료는 **교육용 가상 데이터**이며 실제 위치·수송 명령·인증과 구분합니다.


In [ ]:
#@title 실습자료 자동 준비 (클릭하여 실행) { display-mode: "form" }
from pathlib import Path
import io, zipfile, base64, hashlib, sys, os

# 교육용 MOV06 실습자료가 Notebook 안에 포함되어 있습니다.
# 수동 ZIP 업로드 없이 이 셀을 실행하세요.
_EMBEDDED_ZIP_BASE64 = (
    '...'
)
_EXPECTED_SHA256 = "c0ffcb9baba053edbcf865ea3217588dc9310f836b07a91eae2a8652099511aa"
_archive_bytes = base64.b64decode("".join(_EMBEDDED_ZIP_BASE64), validate=True)
if hashlib.sha256(_archive_bytes).hexdigest() != _EXPECTED_SHA256:
    raise RuntimeError("내장된 실습 자료의 무결성 검사에 실패했습니다. Notebook을 다시 받아주세요.")

destination = Path("/content").resolve()
ROOT = destination / "MOV06_project"
if ROOT.exists():
    if not ROOT.is_dir():
        raise RuntimeError("기존 MOV06_project 경로가 폴더가 아닙니다.")
    print("기존 실습 폴더를 재사용합니다. 수정 코드·기록은 덮어쓰지 않습니다.")
else:
    with zipfile.ZipFile(io.BytesIO(_archive_bytes)) as z:
        bad_file = z.testzip()
        if bad_file is not None:
            raise RuntimeError(f"실습 ZIP 파일 손상: {bad_file}")
        for entry in z.infolist():
            name = entry.filename
            parts = Path(name).parts
            target = (destination / name).resolve()
            if not parts or parts[0] != "MOV06_project" or not target.is_relative_to(destination):
                raise ValueError("실습 ZIP 경로를 확인하세요: " + name)
            if ((entry.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError("심볼릭 링크는 추출하지 않습니다: " + name)
        z.extractall(destination)
    print("MOV06 실습 자료를 자동 준비했습니다. 수동 ZIP 업로드는 필요하지 않습니다.")
if not (ROOT / "pipeline" / "run_all.py").is_file():
    raise RuntimeError("실습 폴더 구성이 올바르지 않습니다. 기존 폴더를 확인하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy, pandas, scipy, sklearn, matplotlib
print("실습 폴더:", ROOT)
print("라이브러리:", {m.__name__:m.__version__ for m in (numpy,pandas,scipy,sklearn,matplotlib)})
from pipeline.run_all import run_mission


라이브러리가 없을 때만 다음 설치 셀의 주석을 풀고 실행 후 런타임을 다시 시작하세요.

In [ ]:
# 필요할 때만 실행
# %pip install numpy pandas scipy scikit-learn matplotlib

## 구현 함수 저장

각 미션의 함수 정의를 AI와 작성하고 전체 def를 implementation 문자열에 넣습니다. 이름·인자·기본값을 유지합니다. 저장 시 이전 파일을 .before로 보존합니다. 미구현 함수는 NotImplementedError로 중단됩니다.


In [ ]:
from tools.patch_function import replace_function
from pipeline.run_all import run_mission
import json
TASKS=json.loads((ROOT/"implementation_tasks.json").read_text())
SELECTED_MISSIONS=list(range(1,11)) # 원본 선택 미션 M4·M6·M8 미수행 시 제외하고 양식에 사유 기록


## 작업 기록·백업·복원

기본 준비 셀과 바로 위 **구현 함수 저장 도구** 셀을 실행한 뒤 사용합니다. **수정 코드·양식·문서·실행 기록**을 JSON으로 내려받거나 다시 복원할 수 있습니다. 원본 대용량 데이터는 내장 ZIP에서 자동 준비됩니다. 복원은 기본적으로 기존 작성 파일을 덮어쓰지 않습니다.


In [ ]:
#@title 작업 기록 · JSON 백업/복원 도구 준비 { display-mode: "form" }
from pathlib import Path
import json, base64, hashlib, datetime, re, os, zipfile

# 학생별 실습 내용을 원본 폴더에 보존합니다. 실행 성공/교관 승인을 자동 판정하지 않습니다.
BACKUP_AREAS = ('src', 'pipeline', 'tools', 'forms', 'docs', 'reports', 'notebooks', 'outputs')
def _safe_file(rel):
    p = Path(rel)
    if not rel or p.is_absolute() or '..' in p.parts or '\\' in rel or p.parts[0] not in BACKUP_AREAS:
        raise ValueError('허용되지 않은 백업 경로: '+str(rel))
    full = ROOT/p
    if not full.resolve().is_relative_to(ROOT.resolve()):
        raise ValueError('실습 폴더를 벗어날 수 없습니다: '+str(rel))
    return full

def save_mission_note(mission, decision, ai_conversation, evidence):
    """실제 본인의 판단, AI 질문/답변 원문, 검증 근거를 저장합니다."""
    m = int(mission)
    if m not in range(1,11): raise ValueError('미션 번호는 1～10입니다')
    if not all(isinstance(x,str) and x.strip() for x in (decision,ai_conversation,evidence)):
        raise ValueError('판단, AI 대화 원문, 실제 검증 근거를 작성하세요')
    target = ROOT/'outputs'/'learner_notes'/f'M{m:02d}.md'
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        old=target.with_suffix('.md.before_'+stamp)
        old.write_bytes(target.read_bytes())
    content=f'# M{m:02d} 학생 실습 기록\n\n## 본인 판단\n{decision}\n\n## AI 요청·답변 원문\n{ai_conversation}\n\n## 실제 검증 근거 및 미확인 사항\n{evidence}\n'
    target.write_text(content,encoding='utf-8')
    print('기록 저장:',target.relative_to(ROOT),'※ 학습 PASS 및 교관 승인이 아닙니다')
    return target

def export_work_json():
    """수정 코드, 문서, 미션 출력과 기록을 JSON으로 보존합니다. 데이터셋은 원본 실습 ZIP을 사용합니다."""
    records={}
    for folder in BACKUP_AREAS:
        base=ROOT/folder
        if not base.is_dir():continue
        for file in base.rglob('*'):
            if not file.is_file() or file.is_symlink() or '__pycache__' in file.parts or file.suffix in ('.pyc',):continue
            rel=file.relative_to(ROOT).as_posix()
            raw=file.read_bytes()
            records[rel]={'sha256':hashlib.sha256(raw).hexdigest(),'data_base64':base64.b64encode(raw).decode('ascii')}
    doc={'schema':'MOV06_student_work_backup_v1','created_at':datetime.datetime.now().isoformat(),
         'note':'원본 data 파일은 자동 준비 셀의 내장 ZIP에서 재생성. 이 JSON은 사용자 수정 코드·문서·기록의 보존용이며 PASS 증거가 아님.',
         'files':records}
    out=ROOT.parent/'MOV06_work_backup.json'
    out.write_text(json.dumps(doc,ensure_ascii=False,indent=2),encoding='utf-8')
    print('JSON 백업:',out,'파일 수:',len(records))
    try:
        from google.colab import files
        files.download(str(out))
    except ImportError: pass
    return out

def restore_work_json(path=None, overwrite=False):
    """기본은 기존 수정 파일을 덮어쓰지 않습니다. 덮어쓰기는 overwrite=True를 직접 지정해야 합니다."""
    source=Path(path) if path is not None else ROOT.parent/'MOV06_work_backup.json'
    doc=json.loads(source.read_text(encoding='utf-8'))
    if doc.get('schema')!='MOV06_student_work_backup_v1' or not isinstance(doc.get('files'),dict):
        raise ValueError('MOV06 작업 백업 JSON이 아닙니다')
    prepared=[]
    for rel,info in doc['files'].items():
        target=_safe_file(rel)
        raw=base64.b64decode(info['data_base64'],validate=True)
        if hashlib.sha256(raw).hexdigest()!=info['sha256']:raise ValueError('백업 파일 해시가 다릅니다: '+rel)
        prepared.append((target,raw))
    restored=skipped=conflicts=0
    for target,raw in prepared:
        if target.exists():
            if target.read_bytes()==raw:skipped+=1;continue
            if not overwrite:conflicts+=1;continue
            stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            target.with_name(target.name+'.before_restore_'+stamp).write_bytes(target.read_bytes())
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(raw);restored+=1
    print(f'복원 완료: {restored}개 / 동일하여 건너뜀: {skipped}개 / 보호한 충돌: {conflicts}개')
    if conflicts: print('기존 작성 파일 보호: 백업으로 교체하려는 경우에만 overwrite=True를 사용하세요')
    return {'restored':restored,'unchanged':skipped,'protected_conflicts':conflicts}

print('작업 기록·백업·복원 도구 준비 완료')


In [ ]:
#@title M10 환경 사전점검 (결과 기록, 비파괴) { display-mode: "form" }
from pathlib import Path
import ast
project_root = next((x for x in [Path('/content/MOV06_project'), Path('/MOV06_project'), Path.cwd()/'MOV06_project'] if x.is_dir()), Path('/content/MOV06_project'))
print('실습 폴더:', project_root, '존재:', project_root.is_dir())
if project_root.is_dir():
    print('미션별 양식:', (project_root/'forms'/'MOV06_M10_forms.md').exists())
    print('파이프라인:', (project_root/'pipeline'/'run_all.py').exists())
else:
    print('실습자료 자동 준비 셀을 먼저 실행하세요.')
print('이 검사는 학습 PASS나 교관 승인을 뜻하지 않습니다.')


## 10. 수행 전후 점검과 제출물

### 구현·검증 기준
미션 10 평가 안내 원본 기준 필수 미션 입니다. 정량 평가, 정성 평가, 진행·완료 확인을 서로 구분합니다. 아래 항목은 실제 수행한 증거를 정리하기 위한 질문이며 원본 평가 수치와 제출 조건을 변경하지 않습니다. 원본 정량 평가 기준 [필수] 재학습 파이프라인·문서 완비 — 자동 재학습 파이프라인이 오류 없이 실행되고 운영 문서 100% 완비, 자체 점검 전 항목 PASS이며, 경보 건이 자동 집행되지 않고 7일 납품 소요 예측이 기준보다 정확하게 브리핑으로 정리되었는가. 활동별 확인 질문 1단계 — 재학습 시간선과 통합 점검 계획: 사용할 입력·기간·단위와 검증 계획을 설명하고, 본인의 판단과 AI 역할을 구분했나요? 2단계 — 재학습 함수와 드리프트 비교 구현하기: 핵심 함수를 실제로 구현·저장하고, 첫 실행 결과와 변경 후 재실행 기록을 남겼나요? 3단계 — 독립 평가·승인 대기·소요 예측 검증하기: 원본 평가 기준을 실제 출력·기간·분모와 대조하고, 오류·한계 및 AI 제안 3건 이상을 검토했나요? 4단계 — 운영 문서 12종과 브리핑 완성하기: 지정 산출물과 작성 양식에 실제 결과, AI 대화 원문 위치, 수정 이유와 최종 판단을 남겼나요? 판정 시 유의사항 각 활동의 자세한 PASS 근거는 위 학습 활동 및 평가 기준 세 항목을 따릅니다. 목표 미달은 미달로, 미측정은 미측정으로 남깁니다. 코드 실행 성공·제출 ZIP 생성·학습 목표 달성·교관 승인은 서로 다른 상태입니다. 출제자 Solution은 학생에게 제공하지 않습니다.

### 작성해야 할 산출물
- `pipeline/mov06_retrain.py`
- `docs/mov06_operations_guide.md`
- `docs/mov06_failure_guide.md`
- `docs/mov06_interface_spec.md`
- `reports/mov06_briefing.md`
- `forms/MOV06_M10_forms.md`

### 확인 순서
1. 교육용 프로젝트의 실제 소스 파일에 함수를 저장하고 변경 전 사본을 확인합니다.
2. 해당 미션 실행 결과의 로그·표·그림을 읽고 코드 실행 결과와 성능 목표를 분리해 기록합니다.
3. 변경 과제를 같은 비교 조건으로 재실행한 뒤 수치·한계를 확인합니다.
4. Forms에 실제 AI 요청과 답변 원문, 근거 대조 3건 이상, 본인의 판단을 기록합니다.
5. JSON 백업과 검토용 ZIP을 보관합니다. 생성 여부는 평가 통과나 실제 교관 승인을 뜻하지 않습니다.


## M10 군수품 이동 상태 추적 및 도착 예정 시간(ETA) 예측 시스템 최종 완성

코드: `pipeline/mov06_retrain.py`

변경 과제: drift_mape를 8.0과 5.0으로 비교하고 드리프트 판단과 3% 승격 정책을 구분한다. 신규 40개와 독립 평가 20개 시간선, 승인 대기, 7일 소요 예측과 운영 문서 12종 본문을 완성한다.

기존 학습 0～199, 추가 학습 200～229, 모델 선택 230～239, 독립 최종 평가 240～259로 총 260개 이동을 사용합니다. 원본 신규 40개 이동(200～239)에 최종 검증용 20개를 추가한 출제자 보완입니다. 드리프트와 챌린저 승격은 별도 판정이며 원본은 드리프트가 없어도 후보를 학습합니다. 문서 12종은 실제 파일·본문·미작성 표시를 확인합니다. 초기 문서 FAIL은 학습자가 작성한 후 재실행합니다. 미래 7일 예측에는 실제값이 제공되지 않으므로 그 정확도를 확정하지 않습니다.

forms/MOV06_M10_forms.md에 실제 계획·대화·검증·결과를 기록합니다.

### 구현할 함수

old_end,new_end,hold_n,drift_mape,min_gain → champ_mape/drift/champ_mae/chall_mae/promote/model 사전. 챌린저 학습은 보류 구간 이전까지만 하고 최종 평가 이동은 재학습에서 제외한다.

아래 틀에 전체 함수를 넣어 저장한 뒤 실행 셀로 넘어갑니다.


In [ ]:
# implementation = """
# def retrain_pipeline(old_end, new_end, hold_n=10, drift_mape=8.0, min_gain=0.03):
#     여기에 실제 구현 작성
# """
# backup=replace_function(ROOT/'pipeline/mov06_retrain.py','retrain_pipeline',implementation)
# print("이전 코드 보존:",backup)


In [ ]:
result=run_mission(10)
print(result)
from IPython.display import display,Image
for figure in result["figures"]:display(Image(filename=str(ROOT/figure)))

## 이 미션 작업 상태와 임시 보관
먼저 실행 결과와 AI 대화를 원본 양식에 저장합니다. 아래 셀은 **검토용 ZIP만 생성**하며 합격·교관 승인을 뜻하지 않습니다.


In [ ]:
#@title M10 검토용 ZIP 만들기 (학생 실습 작업 보관)
from pathlib import Path
import zipfile, datetime
_r = Path(ROOT)
assert _r.is_dir(), "실습자료 자동 준비 셀부터 실행하세요."
_folder = _r.parent / "mission_review_archives"
_folder.mkdir(parents=True, exist_ok=True)
_out = _folder / "MOV06_M10_review.zip"
with zipfile.ZipFile(_out, "w", compression=zipfile.ZIP_DEFLATED) as _z:
    for _p in sorted(_r.rglob("*")):
        if _p.is_file() and "__pycache__" not in _p.parts:
            _z.write(_p, _p.relative_to(_r.parent))
print("검토용 ZIP:", _out, "(학습 PASS/교관 승인 아님)")
try:
    from google.colab import files
    files.download(str(_out))
except ImportError:
    print("Colab 외부 실행: 생성 경로에서 파일을 찾으세요.")
